<a href="https://colab.research.google.com/github/ywang0408/LASA/blob/main/src/lasa_pilot/City_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LASA member registration model (training notebook)

Designed for **Google Colab**. Trains member-level **logistic regression** on `LASA2017 - LASA2026.xlsx` and sums predicted probabilities into expected paid registrations per congress.

**Colab quick start**
1. Open this notebook in Colab.
2. Upload `LASA2017 - LASA2026.xlsx` with the **Files** panel (left sidebar → folder icon → upload), **or** run the setup cell and pick the file when prompted.
3. **Runtime → Run all**.

Do not publish the member workbook in a public repository.

**Label:** paid registration is `Registered == 1` and `Reg. Amount > 0`. Lima **2017** is included. Paris **2026** is excluded from training. Keep the member workbook on the machine that runs this notebook.

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

XLSX_NAME = "LASA2017 - LASA2026.xlsx"

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl", "joblib"],
        check=True,
    )
    WORK_DIR = Path("/content")
    XLSX_PATH = WORK_DIR / XLSX_NAME
    ARTIFACT_DIR = WORK_DIR / "lasa_artifacts"
    if not XLSX_PATH.is_file():
        from google.colab import files  # type: ignore

        print(
            f"No {XLSX_NAME} in /content yet. "
            "Upload it in the Files panel first, or choose it below."
        )
        uploaded = files.upload()
        if XLSX_NAME not in uploaded:
            raise FileNotFoundError(f"Expected uploaded file named {XLSX_NAME!r}")
        XLSX_PATH.write_bytes(uploaded[XLSX_NAME])
    REPO_ROOT = None
else:
    REPO_ROOT = None
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / XLSX_NAME).is_file():
            REPO_ROOT = candidate
            break
        if (candidate / "README.md").is_file() and (candidate / "src" / "lasa_pilot").is_dir():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Could not find repo root. Open this notebook from the LASA project folder."
        )
    XLSX_PATH = REPO_ROOT / XLSX_NAME
    if not XLSX_PATH.is_file():
        raise FileNotFoundError(f"Missing {XLSX_PATH}")
    ARTIFACT_DIR = REPO_ROOT / "src" / "lasa_pilot" / "artifacts"
    sys.path.insert(0, str(REPO_ROOT / "src"))

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Running in Colab:", IN_COLAB)
print("Excel:", XLSX_PATH, "→", XLSX_PATH.is_file())
print("Artifacts:", ARTIFACT_DIR)
if REPO_ROOT is not None:
    print("Repo root:", REPO_ROOT)

In [ ]:
from __future__ import annotations

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from typing import Callable
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from lasa_pilot.tools import haversine_km
except ImportError:
    import math

    _COORDINATES = {
        "barcelona": (41.39, 2.17),
        "buenos aires": (-34.60, -58.38),
        "rome": (41.90, 12.50),
        "berlin": (52.52, 13.40),
        "guatemala city": (14.63, -90.51),
        "boston": (42.36, -71.06),
        "lima": (-12.05, -77.04),
        "guadalajara": (20.66, -103.35),
        "montreal": (45.50, -73.57),
        "new york": (40.71, -74.01),
        "mexico city": (19.43, -99.13),
        "sao paulo": (-23.55, -46.63),
        "madrid": (40.42, -3.70),
        "vancouver": (49.28, -123.12),
        "bogota": (4.71, -74.07),
        "san francisco": (37.77, -122.42),
        "paris": (48.86, 2.35),
    }

    def _norm_city(city: str) -> str:
        return " ".join(city.strip().lower().split())

    def haversine_km(origin: str, destination: str) -> float:
        lat1, lon1 = _COORDINATES[_norm_city(origin)]
        lat2, lon2 = _COORDINATES[_norm_city(destination)]
        radius = 6371.0
        phi1, phi2 = math.radians(lat1), math.radians(lat2)
        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)
        a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
        return round(2 * radius * math.asin(math.sqrt(a)), 1)

## Congress metadata

Sheet names in the workbook are the source of host city and format (in-person vs virtual).

In [ ]:
CONGRESS_BY_YEAR: dict[int, dict[str, object]] = {
    2017: {"host_city": "Lima", "host_country": "Peru", "in_person": True},
    2018: {"host_city": "Barcelona", "host_country": "Spain", "in_person": True},
    2019: {"host_city": "Boston", "host_country": "USA", "in_person": True},
    2020: {"host_city": "Guadalajara", "host_country": "Mexico", "in_person": False},
    2021: {"host_city": "Virtual", "host_country": "Virtual", "in_person": False},
    2022: {"host_city": "Virtual", "host_country": "Virtual", "in_person": False},
    2023: {"host_city": "Vancouver", "host_country": "Canada", "in_person": True},
    2024: {"host_city": "Bogota", "host_country": "Colombia", "in_person": True},
    2025: {"host_city": "San Francisco", "host_country": "USA", "in_person": True},
    2026: {"host_city": "Paris", "host_country": "France", "in_person": True},
}

# Representative city for great-circle distance (must exist in lasa_pilot.tools.COORDINATES).
COUNTRY_TO_CITY: dict[str, str] = {
    "UNITED STATES": "New York",
    "CANADA": "Montreal",
    "MEXICO": "Mexico City",
    "BRAZIL": "Sao Paulo",
    "ARGENTINA": "Buenos Aires",
    "CHILE": "Buenos Aires",
    "COLOMBIA": "Mexico City",
    "ECUADOR": "Lima",
    "PERU": "Lima",
    "SPAIN": "Madrid",
    "FRANCE": "Paris",
    "UNITED KINGDOM": "London",
}

DEFAULT_ORIGIN_CITY = "Mexico City"
TRAINING_YEARS = tuple(y for y in sorted(CONGRESS_BY_YEAR) if y != 2026)
USE_PAID_LABEL = True  # False => Registered == 1 only
USD_PER_KM = 0.10

## Load and normalize member rows

In [ ]:
def normalize_country(value: object) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "UNKNOWN"
    return " ".join(str(value).strip().upper().split())


def reg_amount_column(columns: list[str]) -> str:
    for name in columns:
        if "amount" in name.lower():
            return name
    raise ValueError(f"No registration amount column in {columns}")


def load_member_table(path: Path) -> pd.DataFrame:
    xl = pd.ExcelFile(path)
    chunks: list[pd.DataFrame] = []
    for sheet in xl.sheet_names:
        raw = pd.read_excel(path, sheet_name=sheet)
        amt_col = reg_amount_column(list(raw.columns))
        frame = raw.rename(columns={amt_col: "reg_amount"}).copy()
        frame = frame[[c for c in frame.columns if c in {
            "Member Number", "Cong. Year", "Country", "Submitted", "Accepted",
            "Rejected", "Registered", "reg_amount",
        }]]
        frame["sheet_name"] = sheet
        chunks.append(frame)
    out = pd.concat(chunks, ignore_index=True)
    out["year"] = out["Cong. Year"].astype(int)
    out["member_number"] = out["Member Number"].astype(int)
    out["country"] = out["Country"].map(normalize_country)
    out["reg_amount"] = pd.to_numeric(out["reg_amount"], errors="coerce").fillna(0.0)
    out["registered"] = (out["Registered"].fillna(0).astype(int) == 1).astype(int)
    if USE_PAID_LABEL:
        out["label"] = ((out["registered"] == 1) & (out["reg_amount"] > 0)).astype(int)
    else:
        out["label"] = out["registered"]
    for year, meta in CONGRESS_BY_YEAR.items():
        mask = out["year"] == year
        out.loc[mask, "host_city"] = meta["host_city"]
        out.loc[mask, "host_country"] = meta["host_country"]
        out.loc[mask, "in_person"] = bool(meta["in_person"])
    return out


members = load_member_table(XLSX_PATH)
members = members[members["year"].isin(TRAINING_YEARS)].copy()
print(members.shape)
members.groupby("year")["label"].agg(["count", "sum", "mean"]).round(3)

## Features (history uses prior congresses only)

In [ ]:
def origin_city_for_country(country: str) -> str:
    return COUNTRY_TO_CITY.get(country, DEFAULT_ORIGIN_CITY)


def travel_cost_usd(host_city: str, country: str, usd_per_km: float) -> float:
    if host_city == "Virtual":
        return 0.0
    origin = origin_city_for_country(country)
    try:
        km = haversine_km(origin, host_city)
    except KeyError:
        km = haversine_km(DEFAULT_ORIGIN_CITY, host_city)
    return round(km * usd_per_km, 2)


def add_history_features(df: pd.DataFrame) -> pd.DataFrame:
    ordered = df.sort_values(["member_number", "year"]).copy()
    grouped = ordered.groupby("member_number", sort=False)
    ordered["prior_registrations"] = grouped["label"].cumsum() - ordered["label"]
    ordered["prior_submissions"] = grouped["Submitted"].cumsum() - ordered["Submitted"]
    ordered["prior_acceptances"] = grouped["Accepted"].cumsum() - ordered["Accepted"]
    ordered["ever_registered_before"] = (ordered["prior_registrations"] > 0).astype(int)
    ordered["travel_cost_usd"] = [
        travel_cost_usd(host, country, USD_PER_KM)
        for host, country in zip(ordered["host_city"], ordered["country"], strict=True)
    ]
    ordered["in_person_int"] = ordered["in_person"].astype(int)
    return ordered


model_df = add_history_features(members)
FEATURE_COLUMNS = [
    "country",
    "host_city",
    "host_country",
    "in_person_int",
    "prior_registrations",
    "prior_submissions",
    "prior_acceptances",
    "ever_registered_before",
    "travel_cost_usd",
]
TARGET = "label"
model_df[FEATURE_COLUMNS].head()

## Model pipelines (logistic regression vs random forest)

In [ ]:
CATEGORICAL = ["country", "host_city", "host_country"]
NUMERIC = [
    "in_person_int",
    "prior_registrations",
    "prior_submissions",
    "prior_acceptances",
    "ever_registered_before",
    "travel_cost_usd",
]


def make_preprocessor(*, scale_numeric: bool) -> ColumnTransformer:
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", min_frequency=25),
                CATEGORICAL,
            ),
            ("num", numeric_step, NUMERIC),
        ]
    )


def make_logreg_pipeline() -> Pipeline:
    clf = LogisticRegression(
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=42,
    )
    return Pipeline([("preprocess", make_preprocessor(scale_numeric=True)), ("clf", clf)])


def make_rf_pipeline() -> Pipeline:
    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    return Pipeline([("preprocess", make_preprocessor(scale_numeric=False)), ("clf", clf)])


def congress_totals(frame: pd.DataFrame) -> pd.Series:
    return frame.groupby("year")[TARGET].sum().astype(int)


def predict_congress_total(pipe: Pipeline, frame: pd.DataFrame) -> pd.Series:
    probs = pipe.predict_proba(frame[FEATURE_COLUMNS])[:, 1]
    scored = frame[["year"]].copy()
    scored["pred"] = probs
    return scored.groupby("year")["pred"].sum()


def run_loocv(pipe_factory: Callable[[], Pipeline], model_name: str) -> pd.DataFrame:
    """Leave-one-congress-out evaluation for one model type."""
    actual_totals = congress_totals(model_df)
    fold_rows: list[dict[str, object]] = []
    for holdout_year in sorted(model_df["year"].unique()):
        train = model_df[model_df["year"] != holdout_year]
        test = model_df[model_df["year"] == holdout_year]
        pipe = pipe_factory()
        pipe.fit(train[FEATURE_COLUMNS], train[TARGET])
        pred_total = float(predict_congress_total(pipe, test)[holdout_year])
        actual_total = int(actual_totals[holdout_year])
        member_auc = roc_auc_score(
            test[TARGET], pipe.predict_proba(test[FEATURE_COLUMNS])[:, 1]
        )
        fold_rows.append(
            {
                "model": model_name,
                "holdout_year": holdout_year,
                "host_city": CONGRESS_BY_YEAR[holdout_year]["host_city"],
                "actual_paid": actual_total,
                "predicted_paid": round(pred_total, 1),
                "abs_error": round(abs(pred_total - actual_total), 1),
                "member_auc": round(member_auc, 3),
            }
        )
    report = pd.DataFrame(fold_rows)
    report["pct_error"] = (
        100 * report["abs_error"] / report["actual_paid"].clip(lower=1)
    ).round(1)
    return report

## Leave-one-congress-out evaluation (compare models)

In [ ]:
actual_totals = congress_totals(model_df)

print("Running LOOCV: logistic regression...")
cv_report_logreg = run_loocv(make_logreg_pipeline, "logistic_regression")
print("Running LOOCV: random forest (may take a few minutes)...")
cv_report_rf = run_loocv(make_rf_pipeline, "random_forest")

cv_report = pd.concat([cv_report_logreg, cv_report_rf], ignore_index=True)

model_summary = (
    cv_report.groupby("model", as_index=False)
    .agg(
        congress_mae=("abs_error", "mean"),
        mean_member_auc=("member_auc", "mean"),
        mean_pct_error=("pct_error", "mean"),
    )
    .round(3)
    .sort_values("congress_mae")
)
best_model_name = model_summary.iloc[0]["model"]
print("\nSummary (lower congress_mae is better for attendance totals):")
print(model_summary.to_string(index=False))
print(f"Best on LOOCV congress MAE: {best_model_name}")
cv_report

## Member-level accuracy, precision, and recall (LOOCV)

Uses **out-of-fold** predictions (one held-out congress per fold). Class labels use the same target as training (`label`). Predictions use **probability ≥ 0.5** as registered (paid). Registration is imbalanced, so read these together with **congress MAE** above.

In [ ]:
from typing import Callable

from sklearn.metrics import accuracy_score, precision_score, recall_score

CLASSIFICATION_THRESHOLD = 0.5


def member_metrics_oof(
    pipe_factory: Callable[[], Pipeline],
    model_name: str,
    threshold: float = CLASSIFICATION_THRESHOLD,
) -> tuple[pd.DataFrame, dict[str, float]]:
    fold_rows: list[dict[str, object]] = []
    y_all: list[int] = []
    pred_all: list[int] = []

    for holdout_year in sorted(model_df["year"].unique()):
        train = model_df[model_df["year"] != holdout_year]
        test = model_df[model_df["year"] == holdout_year]
        pipe = pipe_factory()
        pipe.fit(train[FEATURE_COLUMNS], train[TARGET])
        probs = pipe.predict_proba(test[FEATURE_COLUMNS])[:, 1]
        y_true = test[TARGET].to_numpy()
        y_pred = (probs >= threshold).astype(int)

        fold_rows.append(
            {
                "model": model_name,
                "holdout_year": holdout_year,
                "host_city": CONGRESS_BY_YEAR[holdout_year]["host_city"],
                "accuracy": round(accuracy_score(y_true, y_pred), 3),
                "precision": round(
                    precision_score(y_true, y_pred, zero_division=0), 3
                ),
                "recall": round(recall_score(y_true, y_pred, zero_division=0), 3),
                "positive_rate_actual": round(float(y_true.mean()), 3),
                "positive_rate_predicted": round(float(y_pred.mean()), 3),
            }
        )
        y_all.extend(y_true.tolist())
        pred_all.extend(y_pred.tolist())

    y_arr = np.asarray(y_all, dtype=int)
    pred_arr = np.asarray(pred_all, dtype=int)
    pooled = {
        "accuracy": round(accuracy_score(y_arr, pred_arr), 3),
        "precision": round(precision_score(y_arr, pred_arr, zero_division=0), 3),
        "recall": round(recall_score(y_arr, pred_arr, zero_division=0), 3),
    }
    return pd.DataFrame(fold_rows), pooled


print(
    f"Member-level metrics (LOOCV, threshold={CLASSIFICATION_THRESHOLD})..."
)
fold_logreg, pooled_logreg = member_metrics_oof(
    make_logreg_pipeline, "logistic_regression"
)
fold_rf, pooled_rf = member_metrics_oof(make_rf_pipeline, "random_forest")

member_metrics_by_fold = pd.concat([fold_logreg, fold_rf], ignore_index=True)

member_metrics_summary = pd.DataFrame(
    [
        {
            "model": "logistic_regression",
            "accuracy_pooled_oof": pooled_logreg["accuracy"],
            "precision_pooled_oof": pooled_logreg["precision"],
            "recall_pooled_oof": pooled_logreg["recall"],
            "accuracy_mean_by_fold": round(fold_logreg["accuracy"].mean(), 3),
            "precision_mean_by_fold": round(fold_logreg["precision"].mean(), 3),
            "recall_mean_by_fold": round(fold_logreg["recall"].mean(), 3),
        },
        {
            "model": "random_forest",
            "accuracy_pooled_oof": pooled_rf["accuracy"],
            "precision_pooled_oof": pooled_rf["precision"],
            "recall_pooled_oof": pooled_rf["recall"],
            "accuracy_mean_by_fold": round(fold_rf["accuracy"].mean(), 3),
            "precision_mean_by_fold": round(fold_rf["precision"].mean(), 3),
            "recall_mean_by_fold": round(fold_rf["recall"].mean(), 3),
        },
    ]
)

print("\nSummary (pooled = all OOF rows; mean_by_fold = average across 8 congress folds):")
print(member_metrics_summary.to_string(index=False))
print("\nPer held-out congress:")
member_metrics_by_fold

## Fit final model on all training years and save

In [ ]:
final_logreg = make_logreg_pipeline()
final_rf = make_rf_pipeline()
final_logreg.fit(model_df[FEATURE_COLUMNS], model_df[TARGET])
final_rf.fit(model_df[FEATURE_COLUMNS], model_df[TARGET])

final_pipes = {
    "logistic_regression": final_logreg,
    "random_forest": final_rf,
}
final_pipe = final_pipes[best_model_name]

for name, pipe in final_pipes.items():
    artifact_path = ARTIFACT_DIR / f"member_registration_{name}.joblib"
    joblib.dump(
        {
            "pipeline": pipe,
            "model_name": name,
            "feature_columns": FEATURE_COLUMNS,
            "use_paid_label": USE_PAID_LABEL,
            "training_years": TRAINING_YEARS,
            "usd_per_km": USD_PER_KM,
            "congress_by_year": CONGRESS_BY_YEAR,
            "loocv_summary": model_summary.to_dict(orient="records"),
        },
        artifact_path,
    )
    print("Saved:", artifact_path)

if IN_COLAB:
    from google.colab import files  # type: ignore

    for name in final_pipes:
        files.download(str(ARTIFACT_DIR / f"member_registration_{name}.joblib"))
    print("Downloaded both model files.")

in_sample = pd.DataFrame({"year": actual_totals.index, "actual_paid": actual_totals.values})
in_sample["host_city"] = in_sample["year"].map(lambda y: CONGRESS_BY_YEAR[y]["host_city"])
in_sample["pred_logreg"] = predict_congress_total(final_logreg, model_df).round(1).values
in_sample["pred_rf"] = predict_congress_total(final_rf, model_df).round(1).values
in_sample

## Counterfactual: expected attendance for a candidate host

Uses the **latest member list** (max training year) and swaps host fields + travel cost. This is illustrative until LASA confirms the candidate set and counting rule.

In [ ]:
CANDIDATE_HOSTS = [
    ("Rome", "Italy"),
    ("Berlin", "Germany"),
    ("Buenos Aires", "Argentina"),
    ("Guatemala City", "Guatemala"),
    ("Barcelona", "Spain"),
]

latest_year = max(TRAINING_YEARS)
baseline = model_df[model_df["year"] == latest_year].copy()


def scenario_frame(host_city: str, host_country: str) -> pd.DataFrame:
    scenario = baseline.copy()
    scenario["host_city"] = host_city
    scenario["host_country"] = host_country
    scenario["in_person_int"] = 1
    scenario["travel_cost_usd"] = [
        travel_cost_usd(host_city, country, USD_PER_KM) for country in scenario["country"]
    ]
    return scenario


scenario_rows: list[dict[str, object]] = []
for host_city, host_country in CANDIDATE_HOSTS:
    scen = scenario_frame(host_city, host_country)
    row = {
        "host_city": host_city,
        "host_country": host_country,
        "member_rows_scored": len(scen),
        "reference_year_for_roster": latest_year,
    }
    for name, pipe in final_pipes.items():
        total = float(pipe.predict_proba(scen[FEATURE_COLUMNS])[:, 1].sum())
        row[f"expected_{name}"] = round(total, 1)
    scenario_rows.append(row)

scenario_df = pd.DataFrame(scenario_rows).sort_values(
    f"expected_{best_model_name}", ascending=False
)
scenario_df

## Model evaluation plots

**Leave-one-congress-out (LOOCV)** charts compare **logistic regression** and **random forest**. Lower congress **MAE** is the main metric for attendance totals. Member **ROC** / **calibration** use out-of-fold probabilities.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve

MODEL_STYLES = {
    "logistic_regression": {"color": "#2563eb", "label": "Logistic regression"},
    "random_forest": {"color": "#ea580c", "label": "Random forest"},
}


def collect_oof_probs(pipe_factory: Callable[[], Pipeline], model_name: str) -> pd.DataFrame:
    chunks: list[pd.DataFrame] = []
    for holdout_year in sorted(model_df["year"].unique()):
        train = model_df[model_df["year"] != holdout_year]
        test = model_df[model_df["year"] == holdout_year]
        pipe = pipe_factory()
        pipe.fit(train[FEATURE_COLUMNS], train[TARGET])
        probs = pipe.predict_proba(test[FEATURE_COLUMNS])[:, 1]
        chunk = test[["year", TARGET]].copy()
        chunk["prob"] = probs
        chunk["model"] = model_name
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


print("Building out-of-fold predictions for plots...")
oof = pd.concat(
    [
        collect_oof_probs(make_logreg_pipeline, "logistic_regression"),
        collect_oof_probs(make_rf_pipeline, "random_forest"),
    ],
    ignore_index=True,
)

year_labels = {
    int(row["holdout_year"]): f"{int(row['holdout_year'])}\n({row['host_city']})"
    for _, row in cv_report_logreg.iterrows()
}
years_ordered = sorted(year_labels)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Logistic regression vs random forest (LOOCV)", fontsize=14, y=1.02)

# 1) Actual vs predicted (both models)
ax = axes[0, 0]
all_pred = cv_report["predicted_paid"]
all_actual = cv_report["actual_paid"]
lims = [min(all_actual.min(), all_pred.min()) * 0.9, max(all_actual.max(), all_pred.max()) * 1.05]
ax.plot(lims, lims, "--", color="gray", linewidth=1, label="Perfect prediction")
for model_name, style in MODEL_STYLES.items():
    part = cv_report[cv_report["model"] == model_name]
    ax.scatter(
        part["actual_paid"],
        part["predicted_paid"],
        s=70,
        color=style["color"],
        label=style["label"],
        alpha=0.85,
    )
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Actual paid registrations")
ax.set_ylabel("Predicted paid (sum of probabilities)")
ax.set_title("Congress totals: actual vs predicted")
ax.legend(loc="lower right")

# 2) MAE comparison
ax = axes[0, 1]
mae_vals = [model_summary.set_index("model").loc[m, "congress_mae"] for m in MODEL_STYLES]
bars = ax.bar(
    [MODEL_STYLES[m]["label"] for m in MODEL_STYLES],
    mae_vals,
    color=[MODEL_STYLES[m]["color"] for m in MODEL_STYLES],
)
ax.set_ylabel("Mean absolute error (registrations)")
ax.set_title("LOOCV congress total MAE (lower is better)")
for bar, val in zip(bars, mae_vals, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 40, f"{val:.0f}", ha="center", fontsize=10)

# 3) ROC curves
ax = axes[1, 0]
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Random")
for model_name, style in MODEL_STYLES.items():
    part = oof[oof["model"] == model_name]
    fpr, tpr, _ = roc_curve(part[TARGET], part["prob"])
    mean_auc = cv_report[cv_report["model"] == model_name]["member_auc"].mean()
    ax.plot(fpr, tpr, color=style["color"], linewidth=2, label=f"{style['label']} (AUC {mean_auc:.2f})")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Member-level ROC (out-of-fold)")
ax.legend(loc="lower right")

# 4) Calibration
ax = axes[1, 1]
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1, label="Perfect calibration")
for model_name, style in MODEL_STYLES.items():
    part = oof[oof["model"] == model_name]
    frac_pos, mean_pred = calibration_curve(
        part[TARGET], part["prob"], n_bins=10, strategy="quantile"
    )
    ax.plot(mean_pred, frac_pos, "o-", color=style["color"], label=style["label"])
ax.set_xlabel("Mean predicted probability (bin)")
ax.set_ylabel("Fraction registered (paid)")
ax.set_title("Calibration (member-level, LOOCV)")
ax.legend(loc="upper left")

plt.tight_layout()
eval_fig_path = ARTIFACT_DIR / "model_comparison_loocv.png"
fig.savefig(eval_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", eval_fig_path)

# Per-year predicted totals (both models)
fig2, ax2 = plt.subplots(figsize=(13, 5))
x = np.arange(len(years_ordered))
width = 0.22
actual_by_year = cv_report_logreg.set_index("holdout_year")["actual_paid"]
ax2.bar(x - width, [actual_by_year[y] for y in years_ordered], width, label="Actual", color="#16a34a")
for i, model_name in enumerate(MODEL_STYLES):
    part = cv_report[cv_report["model"] == model_name].set_index("holdout_year")
    offset = width * i
    ax2.bar(
        x + offset,
        [part.loc[y, "predicted_paid"] for y in years_ordered],
        width,
        label=MODEL_STYLES[model_name]["label"],
        color=MODEL_STYLES[model_name]["color"],
    )
ax2.set_xticks(x + width / 2)
ax2.set_xticklabels([year_labels[y] for y in years_ordered], rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("Paid registrations")
ax2.set_title("Held-out year: actual vs predicted totals")
ax2.legend()
plt.tight_layout()
year_fig_path = ARTIFACT_DIR / "model_comparison_by_year.png"
fig2.savefig(year_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", year_fig_path)

# In-sample (both models)
fig3, ax3 = plt.subplots(figsize=(6, 6))
lims3 = [
    min(in_sample["actual_paid"].min(), in_sample[["pred_logreg", "pred_rf"]].min().min()) * 0.9,
    max(in_sample["actual_paid"].max(), in_sample[["pred_logreg", "pred_rf"]].max().max()) * 1.05,
]
ax3.plot(lims3, lims3, "--", color="gray", linewidth=1)
ax3.scatter(in_sample["actual_paid"], in_sample["pred_logreg"], color="#2563eb", label="Logistic regression")
ax3.scatter(in_sample["actual_paid"], in_sample["pred_rf"], color="#ea580c", label="Random forest")
ax3.set_xlim(lims3)
ax3.set_ylim(lims3)
ax3.set_xlabel("Actual paid registrations")
ax3.set_ylabel("Predicted (in-sample)")
ax3.set_title("In-sample fit (optimistic — not for reporting)")
ax3.legend()
plt.tight_layout()
insample_fig_path = ARTIFACT_DIR / "model_comparison_insample.png"
fig3.savefig(insample_fig_path, dpi=150, bbox_inches="tight")
plt.show()

# Candidate scenarios (both models)
fig4, ax4 = plt.subplots(figsize=(9, max(4, 0.55 * len(scenario_df))))
y_pos = np.arange(len(scenario_df))
bar_h = 0.35
ax4.barh(y_pos - bar_h / 2, scenario_df["expected_logistic_regression"], height=bar_h, color="#2563eb", label="Logistic regression")
ax4.barh(y_pos + bar_h / 2, scenario_df["expected_random_forest"], height=bar_h, color="#ea580c", label="Random forest")
ax4.set_yticks(y_pos)
ax4.set_yticklabels(scenario_df["host_city"])
ax4.set_xlabel("Expected paid registrations (counterfactual)")
ax4.set_title(f"Candidate hosts (roster from {latest_year}; illustrative)")
ax4.legend(loc="lower right")
plt.tight_layout()
scenario_fig_path = ARTIFACT_DIR / "candidate_host_comparison.png"
fig4.savefig(scenario_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", scenario_fig_path)

if IN_COLAB:
    from google.colab import files  # type: ignore

    for path in (eval_fig_path, year_fig_path, insample_fig_path, scenario_fig_path):
        files.download(str(path))

## Travel cost and city comparison tables (issues #5 and #7)

Every member gets a home location (their country's capital, or the main population hub for the US, Canada and Brazil), so no one defaults to Mexico City. The first table averages each member's trip to every candidate city under three airfare scenarios (a planning estimate, not a ticket price).

The second table puts LASA's candidate cities side by side: expected paid registrations from both models, member travel cost, and flags. No "best city" ranking: none of these cities has hosted LASA in 2018-2026, so the model has no history for them and differences are indicative. Host feasibility (venue, visas, hotels) is not yet verified.

In [ ]:
# ===== Travel cost + city comparison (issues #5 and #7), all in one cell =====
# Needs Yang's cells above to have run first (Runtime > Run all).

# Part 7, cell 1: home location for every member country
import math  # for the distance formula
import pandas as pd  # tables

# One point per country: its capital, or the main population hub for big countries
COUNTRY_COORDS = {
    "UNITED STATES": (38.5, -92.0),  # rough US population centre (not one city)
    "CANADA": (43.65, -79.38), "MEXICO": (19.43, -99.13), "BRAZIL": (-23.55, -46.63),  # Toronto, Mexico City, Sao Paulo
    "CUBA": (23.11, -82.37), "CHILE": (-33.45, -70.67), "COLOMBIA": (4.71, -74.07),  # Havana, Santiago, Bogota
    "ARGENTINA": (-34.60, -58.38), "PERU": (-12.05, -77.04), "ECUADOR": (-0.18, -78.47),  # Buenos Aires, Lima, Quito
    "URUGUAY": (-34.90, -56.16), "BOLIVIA": (-16.50, -68.15), "PARAGUAY": (-25.26, -57.58),  # Montevideo, La Paz, Asuncion
    "VENEZUELA": (10.48, -66.90), "PUERTO RICO": (18.47, -66.11), "COSTA RICA": (9.93, -84.09),  # Caracas, San Juan, San Jose
    "GUATEMALA": (14.63, -90.51), "EL SALVADOR": (13.69, -89.22), "HONDURAS": (14.07, -87.19),  # Central America capitals
    "NICARAGUA": (12.11, -86.24), "PANAMA": (8.98, -79.52), "BELIZE": (17.50, -88.20),  # Central America capitals
    "DOMINICAN REPUBLIC": (18.49, -69.93), "HAITI": (18.59, -72.31), "JAMAICA": (17.97, -76.79),  # Caribbean
    "TRINIDAD AND TOBAGO": (10.65, -61.52), "BARBADOS": (13.10, -59.61), "BAHAMAS": (25.05, -77.35),  # Caribbean
    "CURACAO": (12.11, -68.93), "MARTINIQUE": (14.61, -61.07), "BERMUDA": (32.29, -64.78),  # Caribbean / Atlantic
    "VIRGIN ISLANDS (U.S.)": (18.34, -64.93), "SURINAME": (5.85, -55.20), "FRENCH GUIANA": (4.92, -52.33),  # Caribbean / South America
    "UNITED KINGDOM": (51.51, -0.13), "SPAIN": (40.42, -3.70), "GERMANY": (52.52, 13.40),  # London, Madrid, Berlin
    "FRANCE": (48.86, 2.35), "ITALY": (41.90, 12.50), "PORTUGAL": (38.72, -9.14),  # Paris, Rome, Lisbon
    "NETHERLANDS": (52.37, 4.90), "BELGIUM": (50.85, 4.35), "SWITZERLAND": (47.37, 8.54),  # Amsterdam, Brussels, Zurich
    "LUXEMBOURG": (49.61, 6.13), "IRELAND": (53.35, -6.26), "ICELAND": (64.15, -21.94),  # Western Europe
    "NORWAY": (59.91, 10.75), "SWEDEN": (59.33, 18.07), "DENMARK": (55.68, 12.57), "FINLAND": (60.17, 24.94),  # Nordics
    "AUSTRIA": (48.21, 16.37), "POLAND": (52.23, 21.01), "CZECH REPUBLIC": (50.08, 14.44),  # Central Europe
    "HUNGARY": (47.50, 19.04), "ROMANIA": (44.43, 26.10), "BULGARIA": (42.70, 23.32),  # Eastern Europe
    "CROATIA": (45.81, 15.98), "SERBIA": (44.79, 20.45), "ALBANIA": (41.33, 19.82), "GREECE": (37.98, 23.73),  # Balkans
    "LITHUANIA": (54.69, 25.28), "ESTONIA": (59.44, 24.75), "UKRAINE": (50.45, 30.52), "RUSSIA": (55.76, 37.62),  # Baltics / East
    "TURKEY": (41.01, 28.98), "CYPRUS": (35.17, 33.36), "GEORGIA": (41.72, 44.79), "ARMENIA": (40.18, 44.51),  # Istanbul etc.
    "ISRAEL": (32.09, 34.78), "LEBANON": (33.89, 35.50), "IRAQ": (33.31, 44.36), "QATAR": (25.29, 51.53),  # Middle East
    "UNITED ARAB EMIRATES": (25.20, 55.27), "EGYPT": (30.04, 31.24), "MOROCCO": (34.02, -6.84), "ALGERIA": (36.75, 3.06),  # Middle East / North Africa
    "SENEGAL": (14.72, -17.47), "GHANA": (5.60, -0.19), "NIGERIA": (6.52, 3.38), "CAMEROON": (3.87, 11.52),  # West / Central Africa
    "KENYA": (-1.29, 36.82), "MOZAMBIQUE": (-25.97, 32.57), "SOUTH AFRICA": (-26.20, 28.05), "CABO VERDE": (14.93, -23.51),  # Africa
    "RÉUNION": (-20.88, 55.45), "INDIA": (28.61, 77.21), "PAKISTAN": (33.68, 73.05), "CHINA": (39.90, 116.40),  # Indian Ocean / Asia
    "HONG KONG": (22.32, 114.17), "TAIWAN": (25.03, 121.57), "JAPAN": (35.68, 139.69), "SOUTH KOREA": (37.57, 126.98),  # East Asia
    "PHILIPPINES": (14.60, 120.98), "VIETNAM": (21.03, 105.85), "THAILAND": (13.76, 100.50), "CAMBODIA": (11.56, 104.93),  # Southeast Asia
    "MALAYSIA": (3.14, 101.69), "SINGAPORE": (1.35, 103.82), "INDONESIA": (-6.21, 106.85), "EAST TIMOR": (-8.56, 125.57),  # Southeast Asia
    "AUSTRALIA": (-33.87, 151.21), "NEW ZEALAND": (-36.85, 174.76), "FRENCH POLYNESIA": (-17.53, -149.57),  # Oceania
    "AMERICAN SAMOA": (-14.28, -170.70),  # Oceania
}

# Different spellings found in the data, mapped to one name
ALIASES = {"UNITED STATED": "UNITED STATES", "UK": "UNITED KINGDOM", "THE NETHERLANDS": "NETHERLANDS",  # typos / short forms
           "CZECHIA": "CZECH REPUBLIC", "TURKIYE": "TURKEY", "U.S. VIRGIN ISLANDS": "VIRGIN ISLANDS (U.S.)"}  # alternative names

# Candidate host cities (LASA's list + Santiago 2028 + Montreal for comparison)
HOST_COORDS = {"Montevideo": (-34.90, -56.16), "Rome": (41.90, 12.50), "Milan": (45.46, 9.19),  # LASA candidates
               "Turin": (45.07, 7.69), "Berlin": (52.52, 13.40), "Madrid": (40.42, -3.70),  # LASA candidates
               "Santiago": (-33.45, -70.67), "Montreal": (45.50, -73.57)}  # 2028 host, extra comparison


def home_coords(country):  # look up a member's home point, or None if unknown
    name = ALIASES.get(country, country)  # fix spelling variants
    return COUNTRY_COORDS.get(name)  # None for blank / unknown countries


def km_between(a, b):  # great-circle distance in km between two (lat, lon) points
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))  # degrees to radians
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2  # haversine formula
    return 2 * 6371 * math.asin(math.sqrt(h))  # Earth radius 6,371 km


# Coverage check: how many members on the latest list have a known home point?
roster = baseline[["member_number", "country"]].copy()  # latest member list from the notebook above
roster["coords"] = roster["country"].map(home_coords)  # attach home point
missing = roster[roster["coords"].isna()]  # members we can't place
print(f"Members on latest list: {len(roster):,}")  # total
print(f"Placed on the map: {len(roster) - len(missing):,} ({1 - len(missing) / len(roster):.1%})")  # coverage
print("Not placed (country blank or unknown):", missing["country"].value_counts().to_dict())  # what's left

# Part 7, cell 2: member-weighted travel cost to each candidate city
SCENARIOS = {"low": 0.08, "mid": 0.10, "high": 0.15}  # airfare assumptions in USD per km
NEAR_KM = 3000  # "short trip" cut-off for the share-of-members column

placed = roster.dropna(subset=["coords"])  # only members with a known home point
rows = []  # one row per city
for city, city_xy in HOST_COORDS.items():  # go through each candidate
    km = placed["coords"].map(lambda xy: km_between(xy, city_xy))  # distance for every member
    row = {"City": city, "Avg distance (km)": km.mean()}  # average trip length
    for name, rate in SCENARIOS.items():  # cost under each airfare scenario
        row[f"Avg cost {name} (${rate:.2f}/km)"] = (km * rate).mean()  # member-weighted average cost
    row[f"% members within {NEAR_KM:,} km"] = (km <= NEAR_KM).mean() * 100  # share with a short trip
    rows.append(row)  # save the row

travel_table = pd.DataFrame(rows).set_index("City").round(0)  # tidy table
print(travel_table)  # show the travel-cost table

# City comparison table: one row per candidate city, side by side (no ranking)
CANDIDATES = {"Montevideo": "Uruguay", "Rome": "Italy", "Milan": "Italy", "Turin": "Italy",  # LASA's list
              "Berlin": "Germany", "Madrid": "Spain", "Santiago": "Chile"}  # plus Santiago (2028 host)

past_hosts = {v["host_city"] for v in CONGRESS_BY_YEAR.values()}  # cities that hosted 2018-2026
try:  # make sure the model's travel feature knows every candidate city
    import lasa_pilot.tools as _tools  # running from the repo
    _coords = _tools.COORDINATES  # repo coordinate table
except ImportError:  # running in Colab
    _coords = _COORDINATES  # notebook's fallback table
for city, xy in HOST_COORDS.items():  # coordinates from the travel-cost section
    _coords[city.lower()] = xy  # add any missing city

rows = []  # one row per city
for city, country in CANDIDATES.items():  # go through each candidate
    scen = scenario_frame(city, country)  # latest member list with this city as host
    row = {"City": city, "Country": country}  # labels
    for name, pipe in final_pipes.items():  # both trained models
        label = "RF" if name == "random_forest" else "LogReg"  # short column name
        row[f"Expected attendance ({label})"] = pipe.predict_proba(scen[FEATURE_COLUMNS])[:, 1].sum()  # sum of chances
    row["Avg travel cost (mid)"] = travel_table.loc[city, "Avg cost mid ($0.10/km)"]  # from the travel-cost table
    row["% members within 3,000 km"] = travel_table.loc[city, "% members within 3,000 km"]  # short-trip share
    row["Hosted before?"] = "yes" if city in past_hosts else "no"  # history check
    row["Confidence"] = "has history" if city in past_hosts else "low: no past congress here"  # same as the repo's flag
    row["Host feasibility"] = "unverified"  # venue, visas, hotels not checked yet
    rows.append(row)  # save the row

comparison = pd.DataFrame(rows).set_index("City").round(0)  # tidy table
print(f"Past average paid registrations: {actual_totals.mean():,.0f}")  # reference point
comparison  # side by side, no 'best city' ranking

## City comparison map (issue #7)

Pick a candidate host city to see the model's **expected paid registrations** and where attendees would come from. Hover over a country for its members, expected attendees, average chance to attend, and average travel cost. Cities that never hosted LASA differ mainly through travel cost, so treat results as **indicative**. This map is interactive and needs the notebook to be run (Runtime > Run all).

In [ ]:
# ============================================================
# LASA HOST-CITY MAP (interactive)
# Pick a host city -> see where expected attendees come from.
#  - Globe view: drag to spin, scroll to zoom
#  - Hover a country for members, expected attendees, avg chance, travel cost
#  - Airfare slider changes the travel-cost scenario and re-predicts
#  - Click legend items to show/hide layers
# Run the whole notebook first (Runtime > Run all), then this cell.
# ============================================================
try:  # Colab needs this to show chart widgets
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except ImportError:
    pass
try:  # newer Plotly versions need anywidget for FigureWidget
    import anywidget  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anywidget"], check=False)

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display, HTML

# ---- cities you can pick (city: country, lat, lon) ----
CITY_OPTIONS = {
    "Montreal": ("Canada", 45.50, -73.57),
    "Montevideo": ("Uruguay", -34.90, -56.16),
    "Rome": ("Italy", 41.90, 12.50),
    "Milan": ("Italy", 45.46, 9.19),
    "Turin": ("Italy", 45.07, 7.69),
    "Berlin": ("Germany", 52.52, 13.40),
    "Madrid": ("Spain", 40.42, -3.70),
    "Santiago": ("Chile", -33.45, -70.67),
    "Barcelona": ("Spain", 41.39, 2.17),
    "Buenos Aires": ("Argentina", -34.60, -58.38),
    "Bogota": ("Colombia", 4.71, -74.07),
    "Mexico City": ("Mexico", 19.43, -99.13),
    "Lima": ("Peru", -12.05, -77.04),
    "Boston": ("USA", 42.36, -71.06),
    "Vancouver": ("Canada", 49.28, -123.12),
}

# approximate country centres, used to draw travel lines (add more if needed)
COUNTRY_CENTRES = {
    "UNITED STATES": (39.8, -98.6), "CANADA": (56.1, -106.3), "MEXICO": (23.6, -102.6),
    "BRAZIL": (-14.2, -51.9), "ARGENTINA": (-38.4, -63.6), "CHILE": (-35.7, -71.5),
    "COLOMBIA": (4.6, -74.3), "PERU": (-9.2, -75.0), "ECUADOR": (-1.8, -78.2),
    "SPAIN": (40.5, -3.7), "FRANCE": (46.2, 2.2), "UNITED KINGDOM": (55.4, -3.4),
    "GERMANY": (51.2, 10.5), "ITALY": (41.9, 12.6), "CUBA": (21.5, -77.8),
    "URUGUAY": (-32.5, -55.8), "PUERTO RICO": (18.2, -66.6), "VENEZUELA": (6.4, -66.6),
    "GUATEMALA": (15.8, -90.2), "COSTA RICA": (9.7, -83.8), "JAPAN": (36.2, 138.3),
    "NETHERLANDS": (52.1, 5.3), "SWITZERLAND": (46.8, 8.2), "BOLIVIA": (-16.3, -63.6),
    "PARAGUAY": (-23.4, -58.4), "DOMINICAN REPUBLIC": (18.7, -70.2), "PORTUGAL": (39.4, -8.2),
    "AUSTRALIA": (-25.3, 133.8), "NICARAGUA": (12.9, -85.2), "HONDURAS": (15.2, -86.2),
    "EL SALVADOR": (13.8, -88.9), "PANAMA": (8.5, -80.8),
}

# make sure every option city has coordinates for the travel-cost feature
try:
    import lasa_pilot.tools as _tools
    _COORD_TABLE = _tools.COORDINATES
except ImportError:
    _COORD_TABLE = _COORDINATES
for _c, (_k, _lat, _lon) in CITY_OPTIONS.items():
    _COORD_TABLE[_c.lower()] = (_lat, _lon)

MODELS = {"Random forest": final_rf, "Logistic regression": final_logreg}
PAST_AVG = float(actual_totals.mean())

# ---- night theme colours ----
OCEAN, LAND, BORDER, BG = "#0b1220", "#1e293b", "#334155", "#0f172a"
HOST, OTHER, TEXT, MUTED = "#f97316", "#38bdf8", "#f1f5f9", "#94a3b8"
SHADE = [[0, "#1e293b"], [0.15, "#7c2d12"], [0.45, "#ea580c"], [0.75, "#fb923c"], [1, "#fde68a"]]

_cache = {}


def predict_city(city, model_name, rate):
    """Expected paid registrations for one host city (cached)."""
    key = (city, model_name, rate)
    if key not in _cache:
        country = CITY_OPTIONS[city][0]
        scen = baseline.copy()
        scen["host_city"], scen["host_country"], scen["in_person_int"] = city, country, 1
        scen["travel_cost_usd"] = [travel_cost_usd(city, c, rate) for c in scen["country"]]
        scen["prob"] = MODELS[model_name].predict_proba(scen[FEATURE_COLUMNS])[:, 1]
        _cache[key] = scen
    scen = _cache[key]
    return float(scen["prob"].sum()), scen


def card(label, value, sub, accent=TEXT):
    return (f"<div style='flex:1;min-width:150px;background:{LAND};border:1px solid {BORDER};"
            f"border-radius:12px;padding:12px 16px'>"
            f"<div style='font-size:11px;color:{MUTED};text-transform:uppercase;letter-spacing:.06em'>{label}</div>"
            f"<div style='font-size:24px;font-weight:700;color:{accent};margin-top:2px'>{value}</div>"
            f"<div style='font-size:12px;color:{MUTED}'>{sub}</div></div>")


def build(city, model_name, view, rate, n_lines):
    """Returns (cards HTML, note HTML, plotly figure) for one selection."""
    total, scen = predict_city(city, model_name, rate)
    host_country, h_lat, h_lon = CITY_OPTIONS[city]
    stats = (scen.groupby("country")
             .agg(members=("prob", "size"), expected=("prob", "sum"),
                  chance=("prob", "mean"), cost=("travel_cost_usd", "mean"))
             .sort_values("expected", ascending=False))
    if "home_coords" in globals():  # use the realistic origins from the travel-cost cell for the hover
        stats["cost"] = [km_between(home_coords(c), (h_lat, h_lon)) * rate if home_coords(c) else float("nan")  # member home -> host
                         for c in stats.index]  # one value per country
    all_totals = {c: predict_city(c, model_name, rate)[0] for c in CITY_OPTIONS}
    rank = sorted(all_totals.values(), reverse=True).index(total) + 1
    diff = total - PAST_AVG
    top = stats.index[0].title()

    # ---- headline cards ----
    cards_html = (
        f"<div style='font-family:Inter,Arial,sans-serif;background:{BG};padding:14px;border-radius:14px'>"
        f"<div style='font-size:22px;font-weight:700;color:{TEXT};margin-bottom:10px'>"
        f"&#9733; {city}, {host_country}</div>"
        f"<div style='display:flex;gap:10px;flex-wrap:wrap'>"
        + card("Expected paid registrations", f"{total:,.0f}", model_name, HOST)
        + card("vs past average", f"{diff:+,.0f}", f"{diff / PAST_AVG * 100:+.1f}% vs {PAST_AVG:,.0f}",
               "#4ade80" if diff >= 0 else "#f87171")
        + card("Rank among options", f"#{rank} / {len(all_totals)}", "by expected registrations")
        + card("Largest source", top, f"{stats['expected'].iloc[0]:,.0f} expected attendees")
        + "</div></div>")

    fig = go.Figure()

    # 1) countries shaded by expected attendees (rich hover)
    fig.add_trace(go.Choropleth(
        locations=[c.title() for c in stats.index], locationmode="country names",
        z=stats["expected"], colorscale=SHADE, marker_line_color=BORDER, marker_line_width=0.4,
        customdata=stats[["members", "chance", "cost"]].to_numpy(),
        hovertemplate=("<b>%{location}</b><br>Expected attendees: <b>%{z:,.0f}</b>"
                       "<br>Members on list: %{customdata[0]:,}"
                       "<br>Avg chance to attend: %{customdata[1]:.0%}"
                       "<br>Avg travel cost: $%{customdata[2]:,.0f}<extra></extra>"),
        colorbar=dict(title=dict(text="Expected<br>attendees", font=dict(color=TEXT)),
                      tickfont=dict(color=MUTED), len=0.55, thickness=12, x=0.98, outlinewidth=0),
        name="Attendees by country"))

    # 2) glowing travel lines from the top N origin countries
    mx = stats["expected"].max()
    shown_legend = False
    for country, row in stats.head(n_lines).iterrows():
        if country not in COUNTRY_CENTRES:
            continue
        o_lat, o_lon = COUNTRY_CENTRES[country]
        w = 1 + row["expected"] / mx * 7
        for width, alpha in ((w * 2.6, 0.12), (w, 0.85)):          # soft glow, then bright core
            core = alpha > 0.5
            fig.add_trace(go.Scattergeo(
                lat=[o_lat, h_lat], lon=[o_lon, h_lon], mode="lines",
                line=dict(width=width, color=f"rgba(249,115,22,{alpha})"),
                hoverinfo="skip", legendgroup="lines",
                showlegend=core and not shown_legend, name="Travel to host"))
            if core:
                shown_legend = True

    # 3) other option cities, sized by expected registrations
    others = [c for c in CITY_OPTIONS if c != city]
    fig.add_trace(go.Scattergeo(
        lat=[CITY_OPTIONS[c][1] for c in others], lon=[CITY_OPTIONS[c][2] for c in others],
        text=others, customdata=[[f"{all_totals[c]:,.0f}", f"{all_totals[c] - total:+,.0f}"] for c in others],  # numbers pre-formatted as text
        mode="markers",
        marker=dict(size=[7 + all_totals[c] / 380 for c in others], color=OTHER, opacity=0.85,
                    line=dict(width=1, color=BG)),
        hovertemplate=("<b>%{text}</b><br>Expected: %{customdata[0]}"
                       "<br>vs selected: %{customdata[1]}<extra></extra>"),
        name="Other host options"))

    # 4) selected host city
    fig.add_trace(go.Scattergeo(
        lat=[h_lat], lon=[h_lon], mode="markers+text", text=[f"<b>{city}</b><br>{total:,.0f}"],
        textposition="top center", textfont=dict(size=14, color=TEXT),
        marker=dict(size=24, color=HOST, symbol="star", line=dict(width=2, color="#fff7ed")),
        hovertemplate=f"<b>{city}</b> (selected)<br>Expected: {total:,.0f}<extra></extra>",
        name="Selected host"))

    geo = dict(showcountries=True, countrycolor=BORDER, showcoastlines=True, coastlinecolor=BORDER,
               showland=True, landcolor=LAND, showocean=True, oceancolor=OCEAN,
               showlakes=False, bgcolor=BG, showframe=False)
    if view == "Globe":
        # tilt the globe so the Americas and the host are both in view
        geo.update(projection_type="orthographic",
                   projection_rotation=dict(lon=(h_lon - 40) if h_lon > -20 else (h_lon + 10), lat=15),
                   lataxis_showgrid=True, lonaxis_showgrid=True,
                   lataxis_gridcolor="#1e293b", lonaxis_gridcolor="#1e293b")
    else:
        geo.update(projection_type="natural earth", projection_rotation=dict(lon=0, lat=0))
    fig.update_geos(**geo)
    fig.update_layout(
        hoverdistance=5,  # dots only grab the mouse when you are right on them
        height=640, paper_bgcolor=BG, plot_bgcolor=BG, margin=dict(l=0, r=0, t=0, b=0),
        font=dict(family="Inter, Arial", color=TEXT, size=12),
        legend=dict(orientation="h", x=0.01, y=0.02, bgcolor="rgba(15,23,42,0.7)",
                    font=dict(color=TEXT), itemclick="toggle"),
        hoverlabel=dict(bgcolor=LAND, bordercolor=HOST, font=dict(color=TEXT, size=13)),
        dragmode="pan")
    note_html = (
        f"<p style='font-family:Arial;font-size:12px;color:{MUTED};background:{BG};"
        f"padding:8px 14px;border-radius:8px;margin-top:6px'>"
        f"Shading = expected attendees by home country (hover for details). Lines = top {n_lines} "
        f"origin countries travelling to the host. Blue dots = other host options, sized by expected "
        f"registrations. Airfare scenario: ${rate:.2f}/km. Estimates use the 2025 member list; cities "
        f"that never hosted LASA differ mainly through travel cost, so read differences as indicative.</p>")
    return cards_html, note_html, fig


# ---- controls ----
style = {"description_width": "90px"}
city_dd = widgets.Dropdown(options=list(CITY_OPTIONS), value="Montreal", description="Host city:", style=style)
model_tg = widgets.ToggleButtons(options=list(MODELS), value="Random forest", description="Model:", style=style)
view_tg = widgets.ToggleButtons(options=["Globe", "Flat map"], value="Globe", description="View:", style=style)
rate_sl = widgets.FloatSlider(value=0.10, min=0.05, max=0.25, step=0.01, readout_format=".2f",
                              description="Airfare $/km:", continuous_update=False, style=style)
lines_sl = widgets.IntSlider(value=12, min=3, max=25, description="Travel lines:",
                             continuous_update=False, style=style)
cards_box = widgets.HTML()  # summary cards
note_box = widgets.HTML()  # explanation under the map
first_cards, first_note, first_fig = build(city_dd.value, model_tg.value, view_tg.value, rate_sl.value, lines_sl.value)
map_widget = go.FigureWidget(first_fig)  # chart widget that Colab can update in place
cards_box.value, note_box.value = first_cards, first_note


def _update(_=None):
    cards, note, fig = build(city_dd.value, model_tg.value, view_tg.value, round(rate_sl.value, 2), lines_sl.value)
    cards_box.value, note_box.value = cards, note  # refresh the cards and the note
    with map_widget.batch_update():  # swap the map contents in one go
        map_widget.data = []  # remove old layers
        map_widget.add_traces(fig.data)  # add the new layers
        map_widget.update_layout(fig.layout.to_plotly_json())  # new view, rotation and colours


for w in (city_dd, model_tg, view_tg, rate_sl, lines_sl):
    w.observe(_update, names="value")

display(widgets.VBox([
    widgets.HBox([city_dd, view_tg]),
    model_tg,
    widgets.HBox([rate_sl, lines_sl]),
    cards_box,
    map_widget,
    note_box,
]))